# Exploring Dates, Times, and Time-Series Data

Dates and times (datetimes), such as the time of a particular sale or the date of a public health statistic, are frequently encountered during preprocessing for data analysis and machine learning. Longitudinal data, or time-series data, is data that is collected repeatedly for the same variables over points in time. **Dates and times data are temporal data that capture when an event occurs**, while **time-series data is widely used to understand patterns, trends, seasonality, and changes over time**. Because machine-learning algorithms generally require numerical and consistently formatted inputs, dates and times often need to be transformed into useful numerical features before they can be used effectively in a model.

In this episode, we will take a practical approach to handling the numerical and temporal aspects of Dates, Times, and Time-Series Data. We will build a toolbox of strategies for converting date and time values into machine-learning-ready features, including working with time zones, selecting observations by date and time, extracting useful components such as year, month, day, and weekday, calculating differences between dates, creating lagged features, and using rolling time windows. Specifically, we will focus on the time-series capabilities of the pandas library, which provides a centralized set of tools for working with temporal data and integrates naturally with Python's general-purpose `datetime` functionality. By the end of the episode, you will be able to prepare, transform, and analyze time-series data while also handling common challenges such as missing observations.

<div class='alert alert-info'>

:::{objectives}
- Convert and manipulate date and time data using pandas datetime functionality.
- Select and filter observations based on dates, times, and time periods.
- Create useful time-based features, including elapsed time, lagged features, and rolling-window statistics.
- Identify and appropriately handle missing values and missing timestamps in time-series data.
:::
</div>

<div class='alert alert-success'>

:::{instructor-note}
- XX minutes teaching
- XX minutes exercising/discussion
:::
</div>

## 1. UCI Individual Household Electric Power Consumption Dataset

The [UCI Individual Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) contains 2,075,259 measurements of electricity consumption from a single household in Sceaux, France, recorded at one-minute intervals from December 2006 to November 2010. It includes date and time information together with numerical measurements such as active power, reactive power, voltage, current intensity, and electricity consumption from three household sub-meterings. The dataset is particularly useful for learning how to handle numerical and time-series data, and it also contains missing values that can be used for data-cleaning exercises.

The analysis starts by importing and inspecting the data, then converting the separate Date and Time columns into a single datetime variable.
We further use `describe()`, `mean()`, `min()`, and `max()` to understand the numerical variables and then group the data by hour to investigate daily electricity-consumption patterns.
This provides a simple introduction to data import, datetime handling, descriptive statistics, grouping, and time-series analysis.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/household_power_consumption.zip"
df = pd.read_csv(url, sep=";", na_values="?")

# quick preview and inspect a dataset's structure, column names, and data types
df.head() # df.tail() # df.sample(5)

In [ ]:
# get number of rows and columns in a DataFrame
df.shape

In [ ]:
# prints a concise summary of a DataFrame
df.info()

In [ ]:
# basic numerical analysis
print(df.describe()) # df.Global_active_power

In [ ]:
# average, minimum and maximum power consumption
print("Average of Global_active_power:", df["Global_active_power"].mean())
print("Minimum of Global_active_power:", df["Global_active_power"].min())
print("Maximum of Global_active_power:", df["Global_active_power"].max())

The analysis starts by importing and inspecting the data, then converting the separate Date and Time columns into a single datetime variable. Students can use `describe()`, `mean()`, `min()`, and `max()` to understand the numerical variables and then group the data by hour to investigate daily electricity-consumption patterns. This provides a simple introduction to data import, datetime handling, descriptive statistics, grouping, and time-series analysis.

## 2. From Date Strings to Datetime

Once we have imported the UCI Individual Household Electric Power Consumption dataset, one of the first preprocessing tasks is to make sure that information representing dates and times is stored in a form that Python and pandas can understand.

In this dataset, the Date (the first column) and Time (the second column) columns initially contain values as **strings**, even though they represent meaningful points in time. The output info tells us that pandas currently sees Date and Time as text rather than datetime information.

In [ ]:
print(df[["Date", "Time"]].head(), '\n')

print(df[["Date", "Time"]].dtypes)

### 2.1 Convert strings to datetime type

Here, we will use `pd.to_datetime()` to convert these strings into pandas datetime values. 

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
print(df.dtypes) # now it is a datetime type

df.head()

Instead of ordinary strings, pandas now recognizes these values as dates. This is important because we can perform operations such as extracting the year, month, and day.

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df.head()

### 2.2 Explicit formats to combine Date and Time

For time-series analysis, having separate Date and Time columns is often less convenient than having one timestamp.

In [ ]:
df["Datetime"] = df["Date"] + pd.to_timedelta(df["Time"])
print(df[["Date", "Time", "Datetime"]].head())

This Datetime column will become especially useful when we later select periods, calculate time differences, create lagged features, and work with rolling windows.

### 2.3 Handling invalid dates

Real-world datasets are rarely perfect. We may encounter values that do not represent valid dates.
```python
dates = pd.Series([
    "16/12/2006",
    "17/12/2006",
    "31/02/2007",
    "18/12/2006"
])
```
The value `31/02/2007` is invalid because February does not have 31 days.

If we attempt `pd.to_datetime(dates, format="%d/%m/%Y")`, pandas can raise a parsing error like `ValueError: day is out of range for month, at position 2` because it cannot interpret the invalid value as a real date.
For safely handle values that cannot be converted, we can use `errors="coerce"`, which can tell pandas to convert invalid dates into missing values.

In [ ]:
dates = pd.Series([
    "16/12/2006",
    "17/12/2006",
    "31/02/2007",
    "18/12/2006"
])

# pd.to_datetime(dates, format="%d/%m/%Y") # leads to "ValueError: day is out of range for month"
pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")

The invalid date becomes `NaT`, which means *Not a Time* and is the datetime equivalent of a missing value such as `NaN`.
This is extremely useful when preprocessing real-world data because one bad date does not necessarily have to stop the entire data-processing pipeline.

We can then identify invalid dates using `converted_dates.isna()` or count them using `converted_dates.isna().sum()`.

In [ ]:
converted_dates = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
print(converted_dates.isna())
print('\nNumber of invalid dates:', converted_dates.isna().sum())

Let's apply the same idea to the UCI dataset, and check how many dates could not be converted.
If the result is greater than zero, we know that some values could not be interpreted as valid dates.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
print(df["Date"].isna().sum())

## 3. Handling Time Zones

After converting the Date and Time columns into a proper Datetime column, the next challenge is understanding **time zones**. A timestamp such as 2006-12-16 17:24:00 tells us the local clock time, but by itself it does not tell us where that time occurred. This distinction becomes particularly important when combining time-series data collected from different locations.

In this section, we will distinguish between naive and timezone-aware datetimes, understand why UTC (Coordinated Universal Time) is commonly used as a standard reference, and learn the difference between localizing a timestamp and converting an already timezone-aware timestamp.

Using the [UCI Individual Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption), we will see how to assign a timezone to our timestamps and then convert them to UTC and other time zones without changing the actual moment represented by the observation.

At the end of the previous section, we created a combined timestamp `Datetime`.

In [ ]:
df[["Date", "Time", "Datetime"]].head()

At this point, our timestamps are **naive datetimes**. A naive datetime contains a date and time but has no timezone information attached to it. We can verify this using `df["Datetime"].dt.tz` and the output should be `None`.

In [ ]:
print(df["Datetime"].dt.tz)

This indicates that pandas knows that "the observation occurred at 17:24". But it does not know "the observation occurred at 17:24 in which timezone?". This distinction may not matter when working exclusively with one local dataset, but it becomes important when we combine datasets from different geographic locations.
For example, when data analysts in Sweden and Greece share observational data for analysis, or even schedule meetings, they should clearly specify which time zone is being used to avoid misunderstandings, scheduling conflicts, or errors in interpreting timestamps.

The timestamp `2006-12-16 17:24:00` could represent `17:24 in London`, `17:24 in Stockholm`, or `17:24 in New York`. These are not the same moment in time.
A timezone-aware datetime contains both timestamp and information about its timezone, such as `2006-12-16 17:24:00+01:00`, in which the `+01:00` tells us that the timestamp is one hour ahead of UTC.

<div class='alert alert-info'>

:::{note}
**UTC** (Coordinated Universal Time) is the primary time standard used worldwide. It provides a common reference point for expressing and comparing times across different time zones, without being affected by daylight saving time.
:::
</div>

### 3.1 Localization: Assigning a local time zone

When assigning a timezone to a naive datetime, it does not change the clock time.

In [ ]:
from datetime import datetime

# get the computer's local timezone
local_now = datetime.now().astimezone()
local_tz = local_now.tzinfo

print(local_now)
print(local_tz)

# if Datetime contains UTC timestamps
df["Datetime_Stockholm"] = (
    df["Datetime"]
    .dt.tz_localize("UTC")
    .dt.tz_convert(local_tz)
)
df[["Date", "Time", "Datetime", "Datetime_Stockholm"]].head()

<div class='alert alert-info'>

:::{note}
Localization is not the same as converting between time zones. So `tz_localize()` means that "This timestamp has no timezone. Assign one."
:::
</div>

### 3.2 Conversion: Changing time zones

What if we want to convert the timestamps to another time zone? The goal is to represent the same point in time using a different local time, without changing the underlying observation or event. This is especially useful when working with data collected across different geographic locations.

We can use `tz_convert()` to convert a timezone-aware timestamp from one time zone to another while preserving the same point in time.

In [ ]:
# convert the same moments to California time
df["Datetime_California"] = df["Datetime_Stockholm"].dt.tz_convert("America/Los_Angeles")

df[["Date", "Time", "Datetime", "Datetime_Stockholm", "Datetime_California"]].head()

After conversion, the clock time changed from 17:24 to 09:24, but the actual moment represented by the timestamp did not change.

This gives us a clean, timezone-aware timestamp that is ready for the next stages of time-series preprocessing, such as selecting dates and times, extracting temporal features, calculating time differences, and creating lagged features.

<div class='alert alert-warning'>

:::{callout} Localization vs. Conversion
- Localization uses `tz_localize()`.
    - When datetime is naive, the clock remains and the output of following code fragment is 17:24.
    ```
    naive = pd.Timestamp("2006-12-16 17:24:00")
    aware = naive.tz_localize("Europe/Paris")
    ```
- Conversion uses `tz_convert()`. 
    - When datetime is already timezone-aware, perform conversion to change the clock time in another timezone.
    ```
    converted = aware.tz_convert("UTC")
    ```
- A common mistake is trying to convert a naive datetime directly `df["Datetime"].dt.tz_convert("UTC")`.
    - This will fail because pandas does not yet know what timezone the naive timestamp represents.
:::
</div>

## 4. Selecting Time-Series Data

Once the UCI Individual Household Electric Power Consumption data has been converted into proper datetime values and, when needed, assigned a timezone, the next step is to select specific dates, times, and time periods efficiently.

In a time-series dataset, we are rarely interested in every observation at once. We may want to examine electricity consumption on a particular day, during a specific month, or across a selected range of dates. The UCI Individual Household Electric Power Consumption dataset contains measurements recorded approximately every minute, producing a large number of observations over several years. This makes date-based selection particularly useful.

In this section, we will use pandas date-based indexing and filtering to navigate the UCI electricity consumption data, learn how a `DatetimeIndex` makes temporal selection much easier, and work with date ranges to extract meaningful subsets of observations.
> These techniques form an important foundation for time-series analysis and will also prepare us for later feature-engineering tasks.

Before selecting dates, it is useful to make sure the data is sorted chronologically via `sort_values("Datetime")`. This is an important step when working with time-series data.

In [ ]:
df = df.sort_values("Datetime")
df[["Date", "Time", "Datetime"]].head()

### 4.1 Select a specific date

When we want to select a specific date, one straightforward approach is to filter the dataframe using a condition. Suppose we want to examine household electricity consumption on December 17, 2006. We can do so by running the code snippet below.

In [ ]:
df_17_dec = df[df["Datetime"].dt.date == pd.Timestamp("2006-12-17").date()]
df_17_dec.head()

We can check how many observations we retrieved via `len(df_17_dec)`. Because the UCI dataset contains approximately one measurement per minute, a complete day should contain close to 24 × 60 = 1,440 observations.

In [ ]:
len(df_17_dec)

Here we are filtering rows based on the value of a datetime column. It works, but pandas provides an even more convenient approach when we use a `DatetimeIndex`. A `DatetimeIndex` tells pandas that the index of the dataframe consists of timestamps.

We can create one by setting our Datetime column as the index `df_index = df.set_index("Datetime")`.

In [ ]:
df_index = df.set_index("Datetime")
df_index.head()

Once `Datetime` is the index, selecting an entire day becomes much simpler. For example, `df_index.loc["2006-12-17"]` tells pandas to give "all observations belonging to December 17 2006". This is considerably more convenient than manually constructing a filtering condition.

In [ ]:
df_index.loc["2006-12-17"]

The DatetimeIndex also allows us to select an entire month using a partial date string.

In [ ]:
print(len(df_index.loc["2007-01"]), '\t', 31*1440)
df_index.loc["2007-01"].head()

> Why is this useful?
Imagine that our analysis asks:
Did household electricity consumption behave differently during December compared with January?
Instead of manually constructing start and end dates, we can select the relevant months directly.

In [ ]:
print(len(df_index.loc["2007"]), '\t', 365*1440)
df_index.loc["2007"].head()

Furthermore, we can select a period rather than a single date, month, or a single year.

In [ ]:
df_period = df_index.loc["2006-12-17":"2006-12-18"]
print(len(df_period))

This is particularly useful for exploring a short section of a much larger time series.

Sometimes we need more precise control than selecting complete days. For example, suppose we want electricity consumption between December 16 2006, 17:30 and 18:00.

In [ ]:
df_half_hour = df_index.loc["2006-12-16 17:30:00":"2006-12-16 18:00:00"]
print(len(df_half_hour))

This gives us observations within that specific time interval. We could then calculate the average power consumption during this period.

In [ ]:
df_index.loc[
    "2006-12-16 17:30:00":"2006-12-16 18:00:00",
    "Global_active_power"
].mean()

This demonstrates why timestamps are much more powerful than ordinary strings: pandas can understand the chronological relationship between them.

We can also use normal Boolean filtering. For example, suppose we want observations from December 2006 where global active power was greater than 7 kW.

In [ ]:
high_power = df_index[
    (df_index.index >= "2006-12-01") &
    (df_index.index < "2007-01-01") &
    (df_index["Global_active_power"] > 7)
]
print(len(high_power))
high_power.head()

This combines two concepts: **Time-based filtering** and **Value-based filtering**. This type of filtering becomes very useful when preparing data for machine-learning analysis.

For the UCI electricity dataset, this means that instead of treating millions of observations as one large table, we can ask much more meaningful questions such as "What happened on this day?", "What happened during December?", or "How much electricity was consumed during the evening?" This provides the foundation for the next step: breaking datetime information into multiple machine-learning features such as year, month, day, hour, and weekday.

## 5. Calculating Differences and Elapsed Time in Time-Series Data

Once we have converted timestamps into proper datetime values and extracted useful calendar features, we can ask another important question: how much time has passed between observations or events? In time-series data, the difference between two timestamps can provide valuable information about the frequency of observations, the duration of an event, or the time since a previous measurement.

In this section, using the UCI Individual Household Electric Power Consumption dataset, we will learn how pandas represents time differences using Timedelta, calculate elapsed time between timestamps, and measure the differences between consecutive electricity-consumption observations. These calculations are useful not only for understanding the structure and quality of a time series but also for creating features that can help machine-learning models capture temporal behavior.

Let's continue with the UCI electricity dataset from the previous sections.
We first check that we already have a Datetime column and that the data is sorted chronologically.

In [ ]:
df = df.sort_values("Datetime")
df[["Datetime", "Global_active_power"]].head()

### 5.1 Subtracting two datetimes

One of the useful properties of pandas datetime objects is that we can simply subtract them.
For example, let's select two timestamps, and the output is a pandas **Timedelta**, which represents a duration or difference between two points in time.

In [ ]:
time1 = pd.Timestamp("2006-12-16 17:24:00")
time2 = pd.Timestamp("2006-12-16 17:30:00")

difference = time2 - time1
print(difference)

<div class='alert alert-info'>

:::{note}
It should be noted that a Timedelta is different from a datetime.
- A datetime represents a point in time: `2006-12-16 17:24:00`.
- A Timedelta represents an amount of elapsed time: `0 days 00:06:00`.

We can think of it as:
- Datetime → "When?"
- Timedelta → "How long?"
:::
</div>

We can also extracting the number of days, and also the duration in other units, such as the numbers of seconds.

In [ ]:
start = pd.Timestamp("2006-12-16")
end = pd.Timestamp("2006-12-20")

elapsed = end - start
print(elapsed)
print(elapsed.days)
print(elapsed.total_seconds())

<div class='alert alert-info'>

:::{questions} Why can time differences help machine learning?
- Time differences can be useful features when the spacing between observations is meaningful.
- For example, imagine two electricity measurements: "Measurement A → 4.2 kW" and "Measurement B → 5.1 kW". 
- The model may benefit from knowing not only the values but also how long passed between A and B?
- In regularly sampled data such as this electricity dataset, the answer is usually about one minute.
- But in event-based datasets -- such as customer purchases, website visits, or medical events -- the elapsed time can vary considerably.
:::
</div>

## 6. Creating Lagged Features: Using the Past to Predict the Future

In time-series data analysis and machine learning tasks, the most useful information for predicting what happens next is often found in what happened previously. A **lagged feature** stores an earlier observation alongside the current observation, allowing a model to learn relationships between past and future values.

For the UCI Individual Household Electric Power Consumption dataset, for example, the electricity consumption one minute ago may contain useful information for predicting consumption at the current minute or at a future time.

In this section, we will learn what a lag is, create lagged electricity-consumption features using pandas, see how lags support forecasting, and most importantly, understand how to avoid data leakage by ensuring that features used to predict a value are available at the time the prediction is made.

Let's continue with the UCI Individual Household Electric Power Consumption dataset. We always start to chronologically sort observations, and thus these observations have a temporal order..

In [ ]:
df = df.sort_values("Datetime")

df[["Datetime", "Global_active_power"]].head()

### 6.1 Creating a one-step lag

A lag means using a previous observation as a feature for the current observation. Suppose our electricity consumption is
```console
Time        Power
17:24       4.216
17:25       5.360
17:26       5.374
17:27       5.388
```
A one-step lag looks like this:
```console
Current Time     Current Power     Previous Power
17:24             4.216              NaN
17:25             5.360             4.216
17:26             5.374             5.360
17:27             5.388             5.374
```

So when we are at 17:26, the lagged feature tells us what the power consumption was at 17:25.

We can use `.shift()` in pandas. In below code snippet, the `.shift(1)` means to move the values down by one row so that the previous observation becomes a feature for the current observation.

In [ ]:
df["Power_Lag_1"] = (df["Global_active_power"].shift(1))
df[["Datetime", "Global_active_power", "Power_Lag_1"]].head()

The first row contains NaN because there is no previous observation available.

If one previous observation is not enough, we can create several lagged features.

In [ ]:
df["Power_Lag_2"] = (df["Global_active_power"].shift(2))
df["Power_Lag_3"] = (df["Global_active_power"].shift(3))

df[["Datetime", "Global_active_power",
    "Power_Lag_1", "Power_Lag_2", "Power_Lag_3"
]].head()

<div class='alert alert-danger'>

:::{questions} **Why are lagged features useful**?
- Electricity consumption often has temporal dependence. In other words, what happens now may be related to what happened recently.
For example, if household electricity consumption is currently high, the consumption one minute ago may also have been high.
- A model can potentially learn relationships such as:
    ```console
        Power at t-1  → Power at t
        Power at t-2  → Power at t
        Power at t-3  → Power at t
    ```
    where:
    ```console
        t     = current time
        t-1   = one observation ago
        t-2   = two observations ago
        t-3   = three observations ago
    ```
- This is one of the fundamental ideas behind many time-series forecasting approaches.
:::
</div>

### 6.2 Connect lagged features for forecasting

Now let's connect lagged features for forecasting. Suppose our goal is to predict electricity consumption at the next minute, we can define the future value as our target.

One simple approach is to shift the target backward `df["Target_Next_Minute"] = (df["Global_active_power"].shift(-1))`.

In [ ]:
df["Target"] = (df["Global_active_power"].shift(-1))
df[["Datetime", "Global_active_power",
    "Target"
]].head()

This means that we can use information available at time t to predict electricity consumption at time t+1.
We could then use `Power_Lag_1`, `Power_Lag_2`, and `Power_Lag_3` as predictors.
The basic forecasting structure becomes:
```console
Past
 ↓
t-3     t-2     t-1       t       t+1
 |       |       |         |        |
 └───────┴───────┴─────────┘        ↓
     Features                  Prediction
```

### 6.3 Building a simple forecasting dataset

Let's building a simple forecasting dataset by combining these lag features and the target together.

In [ ]:
model_df = df[["Datetime", "Global_active_power"]].copy()

model_df["Lag_3"] = (model_df["Global_active_power"].shift(3))
model_df["Lag_2"] = (model_df["Global_active_power"].shift(2))
model_df["Lag_1"] = (model_df["Global_active_power"].shift(1))
model_df["Target"] = (model_df["Global_active_power"].shift(-1))

model_df.head()

In [ ]:
# for a simple demonstration, we remove rows with missing lag values
model_df = model_df.dropna()
model_df.head()

<div class='alert alert-warning'>

:::{warning} Data leakage issues in time-series machine learning
:class: dropdown

**Data leakage** is one of the most important concepts in time-series machine learning. It occurs when information that would not actually be available at the time of prediction is inadvertently used to train or evaluate a model. Leakage can lead to overly optimistic performance estimates because the model has access to information that would not be available in a real-world forecasting scenario.

For example, suppose we want to predict household electricity consumption at 17:30. At that moment, we may have access to the observations recorded at 17:27, 17:28, and 17:29, which can be used as lagged features to predict the power consumption at 17:30. However, we cannot use the actual power consumption at 17:31 as a feature, because this observation belongs to the future and would not yet be available. Including such information would introduce data leakage and make the model's performance appear better than it would be in practice.

There is another form of leakage that is especially important in time-series problems: **temporal leakage caused by an inappropriate train-test split**.
Suppose we randomly split the observations into training and test sets: `train, test = train_test_split(model_df, test_size=0.2,random_state=42)`. This approach can be problematic for time-series forecasting because observations from the future may end up in the training set, while earlier observations are placed in the test set. Consequently, the model may learn from information that would not have been available when making predictions for the test period. This can result in overly optimistic evaluation metrics.

For time-series forecasting, we normally preserve chronological order when splitting the data. A representative code example is shown below:
```python
split_date = "2010-01-01"
train = model_df[model_df["Datetime"] < split_date]
test = model_df[model_df["Datetime"] >= split_date]
```

In this example, the model is trained on observations before January 1, 2010, and evaluated on observations from that date onward. This better reflects the real-world forecasting scenario, where future observations are not available when making predictions.
:::
</div>

For the UCI electricity dataset, lagged features allow a model to use previous electricity consumption to predict future consumption. But we must preserve the chronological structure of the data and never allow future observations to leak into the predictors. This principle will remain important in the next section, where we use rolling time windows to summarize recent history while carefully ensuring that the current or future target does not accidentally enter the feature calculation.

## 7. Using Rolling Time Windows

Lagged features allow us to look at individual previous observations, but sometimes a single past value is too noisy to provide a reliable picture of recent behavior. **Rolling time windows** solve this problem by summarizing a group of recent observations, for example, calculating the average electricity consumption over the previous 5, 30, or 60 minutes.

In this section, using the UCI Individual Household Electric Power Consumption dataset, we will learn how to calculate rolling mean, sum, minimum, and maximum values, and see how these statistics can smooth short-term fluctuations and reveal local trends in household electricity usage. We will also discuss how rolling features can become useful machine-learning predictors while avoiding the important mistake of including future information in the calculation.

In [ ]:
df = df.sort_values("Datetime")
df[["Datetime", "Global_active_power"]].head()

### 7.1 Creating a rolling window

A rolling window takes a group of consecutive observations and calculates a statistic over that group.
Suppose our electricity consumption is:
```
Time       Power
17:24      4.216
17:25      5.360
17:26      5.374
17:27      5.388
17:28      5.400
```

A 3-observation rolling mean might look like:
```
Time       Power       Rolling Mean
17:24      4.216          NaN
17:25      5.360          NaN
17:26      5.374         4.983
17:27      5.388         5.374
17:28      5.400         5.387
```

At 17:26, the rolling window contains `17:24, 17:25, 17:26`, and pandas calculates the average of those observations, (4.216+5.360+5.374)/3 = 4.983.

A rolling window summarizes recent history rather than looking at only one previous observation.

For the rolling window, the most common rolling statistic is the **rolling mean**. Let's calculate a 5-observation rolling average.

In [ ]:
df["Rolling_Mean_5"] = (
    df["Global_active_power"]
    .rolling(window=5)
    .mean()
)

df[[
    "Datetime",
    "Global_active_power",
    "Rolling_Mean_5"
]].head(7)

The first four rows will contain NaN because five observations are required to calculate the first complete 5-observation window.

Because the UCI dataset is approximately minute-level data, a 30-observation window represents approximately 30 minutes. Then we can compare it with the original measurement.

In [ ]:
df["Rolling_Mean_30"] = (
    df["Global_active_power"]
    .rolling(window=30).mean()
)

df[[
    "Datetime",
    "Global_active_power",
    "Rolling_Mean_30"
]].head(40)

The original series may fluctuate considerably from minute to minute, while the rolling mean will generally be smoother.

We can extend the rolling window to approximately one hour. Different window lengths provide different levels of temporal context. In general, a longer rolling window produces smoother output by reducing the influence of short-term fluctuations, while a shorter window is more responsive to rapid changes in the data.

<div class='alert alert-warning'>

:::{callout} The Window Size Matter!

The size of the rolling window determines how much recent history we summarize.
- A rolling window with 5 observations captures very short-term behavior.
- A rolling window with 60 observations captures approximately one hour of history in this dataset.
- Conceptually:
    - Small window -> More responsive -> More sensitive to noise
    - Large window -> More smoothing -> Less sensitive to short-term changes
    - **This is an important modeling decision**.
:::
</div>

### 7.2 Rolling sum, minimum & maximum

A rolling window does not have to calculate an average. We can calculate the rolling sum.

In [ ]:
df["Rolling_Sum_60"] = (
    df["Global_active_power"]
    .rolling(window=60).sum()
)
df[[
    "Datetime",
    "Global_active_power",
    "Rolling_Sum_60"
]].head(70)

This calculates the sum of the previous 60 observations, including the current observation. If each observation represents approximately one minute, this summarizes approximately one hour of measurements.

However, there is an important domain consideration: `Global_active_power` is measured in kilowatts, so simply summing the readings does not directly give energy in kWh unless we account for the one-minute sampling interval.

In [ ]:
df["Energy_kWh"] = (
    df["Global_active_power"] / 60
)
df["Rolling_Energy_1h"] = (
    df["Energy_kWh"]
    .rolling(window=60)
    .sum()
)

This is a useful opportunity to emphasize that the interpretation of a rolling statistic depends on the units and sampling frequency of the original data.

We can also calculate the minimum value in a rolling window.

df["Rolling_Min_60"] = (
    df["Global_active_power"]
    .rolling(window=60)
    .min()
)
df[[
    "Datetime",
    "Global_active_power",
    "Rolling_Min_60"
]].head(65)


This answers "What was the lowest measured power consumption within the recent 60 observations"?
Rolling minimum can be useful when we want to understand the lower boundary of recent activity.


Similarly, we can calculate the maximum:

df["Rolling_Max_60"] = (
    df["Global_active_power"]
    .rolling(window=60)
    .max()
)
df[[
    "Datetime",
    "Global_active_power",
    "Rolling_Max_60"
]].head(65)

### 7.3 Comparing rolling statistics

Now we have four useful summaries:
Rolling Mean → average recent consumption
Rolling Sum  → accumulated recent measurements
Rolling Min  → lowest recent consumption
Rolling Max  → highest recent consumption

The rolling mean tells us the typical recent level, while the minimum and maximum tell us about the recent range of behavior.

In [ ]:
df["Power_Rolling_Mean_30"] = (
    df["Global_active_power"]
    .rolling(30)
    .mean()
)

df["Power_Rolling_Min_30"] = (
    df["Global_active_power"]
    .rolling(30)
    .min()
)

df["Power_Rolling_Max_30"] = (
    df["Global_active_power"]
    .rolling(30)
    .max()
)

df[[
    "Datetime",
    "Global_active_power",
    "Power_Rolling_Mean_30",
    "Power_Rolling_Min_30",
    "Power_Rolling_Max_30"
]].head(35)

Why do we need rolling means?

One important use of rolling means is smoothing.
Imagine the raw electricity series contains rapid fluctuations: "4.2 → 5.3 → 4.8 → 5.7 → 4.5 → 5.1".
These changes may make it difficult to see the underlying pattern.
A rolling mean provides a smoother representation of recent behavior.
    - The raw measurement represents "What happened at this particular observation"?
    - The rolling mean represents something closer to "What has electricity consumption looked like recently"?

Rolling statistics can provide a model with information about recent context.

> A forecasting model can then potentially learn from both individual past observations and summaries of recent history.

> The central lesson is that lagged features tell us about individual past observations, while rolling features summarize a window of recent history. Rolling means can smooth noisy electricity measurements, while rolling minimums and maximums capture the recent range of behavior. When these statistics are used as machine-learning features, we must always ensure that the window contains only information that would genuinely have been available at prediction time.

## 8. Handling Missing Data in Time Series

Missing data is especially important in time-series analysis because there are actually two different kinds of missingness we need to distinguish: a missing value and a missing timestamp. In the UCI Individual Household Electric Power Consumption dataset, for example, a household's electricity measurement may be unavailable for a particular minute, or an entire period of timestamps may be absent from the dataset. These situations require different solutions.

In this section, we will learn how to detect missing values and missing timestamps, understand when to use forward fill, backward fill, or interpolation, and apply these techniques carefully so that we do not create unrealistic patterns or introduce information from the future into a forecasting model.

> We continue using the UCI Individual Household Electric Power Consumption dataset.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/household_power_consumption.zip"
df = pd.read_csv(url, sep=";", na_values="?")

df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True
)

df = df.sort_values("Datetime")
df = df.set_index("Datetime")

df["Global_active_power"]

### 8.1 Missing values vs. missing timestamps

The first concept learners need to understand is that: **A missing value and a missing timestamp are not the same thing**.

Suppose we have:
```
Time       Power
10:00      2.4
10:01      NaN
10:03      2.8
```
Here, the timestamp 10:01 exists, but the measurement is missing. This is a missing value.
However, the timestamp 10:02 is completely absent. This is a missing timestamp.

The distinction matters because filling a missing value is different from reconstructing a missing observation in the time index.

In [ ]:
# 2. Finding Missing Values
# Let's first check how many missing values exist in the electricity data.
df["Global_active_power"].isna().sum()

In [ ]:
# We can check missing values across all columns:
df.isna().sum()

In [ ]:
# calculate percentage of missing values
missing_percent = (df.isna().mean() * 100)
missing_percent
# This gives us a quick overview of which variables contain missing observations.

In [ ]:
# look at missing observations
missing_power = df[df["Global_active_power"].isna()]
missing_power[["Global_active_power"]].head()

After checking missing observations, we can ask a different question: Are any expected timestamps missing from our dataset?

Because this dataset is approximately recorded at one-minute intervals, we can create the expected one-minute timeline.

In [ ]:
# inspect time differences
time_difference = (df.index.to_series().diff())
time_difference.value_counts().head()
# time_difference

This means that the time difference between consecutive observations is 1 minute for 2,075,258 rows. Therefore this dataset does not have missing timestamps. It contains measurements at a one-minute sampling rate, and all calendar timestamps are present.

### 8.2 Filling missing values

One of the simplest ways to fill missing values is **forward fill**.
Forward fill means using the most recent available observation (**previous value**) to fill the missing value.

Suppose the following observations are recorded:
```console
10:00 → 2.4
10:01 → NaN
10:02 → 2.6
```

Forward fill produces:
```console
10:00 → 2.4
10:01 → 2.4
10:02 → 2.6
```

In [ ]:
df_ffill = df.copy()

df_ffill["Global_active_power"] = (
    df_ffill["Global_active_power"]
    .ffill()
)

# check whether missing values remain
df_ffill["Global_active_power"].isna().sum()

**Backward fill** works in the opposite direction. It uses the **next available observation** to fill a missing value.
For example:
```console
10:00 → 2.4
10:01 → NaN
10:02 → 2.6
```
Backward fill gives:
```console
10:00 → 2.4
10:01 → 2.6
10:02 → 2.6
```

In [ ]:
df_bfill = df.copy()

df_bfill["Global_active_power"] = (
    df_bfill["Global_active_power"]
    .bfill()
)
df_bfill["Global_active_power"].isna().sum()

<div class='alert alert-danger'>

:::{questions} Forward Fill or Backward Fill?
- Forward fill uses the previous value, whereas backward fill uses the next value.
- If only one observation is missing in dataset, either forward fill or backward fill is applicable, but neither approach is automatically better.
- The choice depends on the meaning of the data and the goal of the analysis.
:::
</div>

Instead of copying one value forward or backward, we can estimate a missing value based on surrounding observations. This is called **interpolation**.

For the same set of observations, we can also estimate the missing value by interpolation.
```console
10:00 → 2.4
10:01 → NaN
10:02 → 2.6
```
Linear interpolation estimates "10:01 → 2.5" because 2.5 lies halfway between 2.4 and 2.6.


In [ ]:
df_interp = df.copy()

df_interp["Global_active_power"] = (
    df_interp["Global_active_power"]
    .interpolate(method="linear") # linear interpolation
)
df_interp[["Global_active_power"]].isna().sum()

Interpolation is useful when we believe the missing values should lie somewhere between known observations. Besides "linear", pandas supports several interpolation methods as listed below.

| Method | Description |
| :----: | :---------: |
| "time" | Uses the actual time intervals between observations; useful for time-series data |
| "index" | Uses the numerical values of the index |
| "nearest" | Uses the value from the nearest observation |
| "zero" | Uses the previous value |
| "slinear" | First-order spline interpolation |
| "quadratic" | Quadratic spline interpolation |
| "cubic" | Cubic spline interpolation |
| "polynomial" | Polynomial interpolation; requires order |
| "spline" | Spline interpolation; requires order |

Because this is time-series data, we can also use **time-aware interpolation**. The time method uses the time information represented by the datetime index when estimating missing values.

This is particularly useful when observations are not perfectly equally spaced.

In [ ]:
df_interp_time = df.copy()

df_interp_time["Global_active_power"] = (
    df_interp_time["Global_active_power"]
    .interpolate(method="time")
)
df_interp_time[["Global_active_power"]].isna().sum()

In [ ]:
comparison = pd.DataFrame({
    "Original": df["Global_active_power"],
    "Forward_Fill": df_ffill["Global_active_power"],
    "Backward_Fill": df_bfill["Global_active_power"],
    "Interpolation": df_interp["Global_active_power"],
    "Time_Interpolation": df_interp_time["Global_active_power"]
})

missing_rows = comparison[
    comparison["Original"].isna()
]

missing_rows.head(10)

If the timestamps are equally spaced, linear interpolation and time interpolation usually produce the same values.

<div class='alert alert-danger'>

:::{questions} How about a dataset with a long missing period?

For a dataset with a long missing period, avoid ordinary linear or time interpolation. They may create unrealistic values, especially when the signal changes over time.
There are several recommended approach depending on the length of missing period.
- Short gaps: Use linear or time interpolation.
- Long gaps: Use a model-based method, such as: seasonal averages, interpolation using similar days or hours, time-series forecasting, machine-learning imputation.
- Very long gaps: Consider leaving the values missing or removing that period if reliable reconstruction is not possible.
:::
</div>

<div class='alert alert-warning'>

:::{callout}
When working with time-series data, the first question should not simply be "How do I fill the missing values?" Instead, ask "What exactly is missing?"
```console
                  Missing Data
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
       Missing value       Missing timestamp
             │                   │
       Value is NaN        Row is absent
             │                   │
       ┌─────┼─────┐             │
       ↓     ↓     ↓             ↓
     ffill bfill interpolate    reindex
```

The central lesson is that there is no universally correct imputation method for time-series data. Forward fill assumes the most recent value remains reasonable, backward fill uses the next available observation, and interpolation estimates values between observations. Most importantly, when preparing data for forecasting, the imputation process must respect the temporal direction of the problem so that future information does not leak into the past.
:::
</div>

<div class='alert alert-success'>

:::{exercise}
- Select Bike Sharing Demand dataset or Jena Climate Weather dataset that contains date, time, or time-series information.
    ```python
    import pandas as pd

    url = "https://raw.githubusercontent.com/TeamLab/machine_learning_from_scratch_with_python/master/code/ch8/data/train.csv"
    bike = pd.read_csv(url, parse_dates=["datetime"])
    bike.head()

    url = "https://huggingface.co/datasets/sayanroy058/Jena-Climate/resolve/main/jena_climate_2009_2016.csv"
    jena = pd.read_csv(url)
    jena["Date Time"] = pd.to_datetime(
        jena["Date Time"],
        format="%d.%m.%Y %H:%M:%S"
    )
    jena.head()
    ```
- Inspect the dataset by examining its dimensions, columns, data types, and sample observations.
- Calculate descriptive statistics to understand the distribution and characteristics of the numerical variables.
- Inspect and convert date and time columns into appropriate pandas datetime representations.
- Check for missing values and missing timestamps, and identify gaps or irregularities in the time sequence.
- Extract date and time features, such as year, month, day, hour, quarter, day of year, and day of the week.
- Create time-based features, including elapsed-time variables, lagged features, and rolling-window statistics where appropriate.
- Handle missing observations using suitable techniques such as forward fill, backward fill, or interpolation, while considering the temporal meaning of the data.
- Verify the cleaned dataset by checking data types, missing values, chronological ordering, feature values, and the final dataset structure before using it for machine learning.
:::
</div>

<div class='alert alert-info'>

:::{keypoints}
- Convert date and time strings into datetime objects and work with time zones using `tz_localize()` and `tz_convert()`.
- Select and filter time-series observations using dates, time ranges, and DatetimeIndex.
- Extract useful temporal information and calculate time differences for analysis and feature engineering.
- Create lagged and rolling-window features to capture past observations, trends, and local patterns while avoiding data leakage.
- Detect and handle missing values and missing timestamps using techniques such as reindexing, forward fill, backward fill, and interpolation.
:::
</div>